# 03 · Integración distrital en Gold

Ejecutar en el nodo primario de EMR con el kernel **American Tower - PySpark**, en un kernel nuevo y en orden. Cada celda muestra la evidencia que produce. Los registros de Spark y el resultado quedan enlazados por el identificador de corrida.

Reconstruye el universo distrito-trimestre a partir de las dimensiones Silver. Integra la serie de velocidad y publica
el contexto territorial fechado en una tabla separada. Se verifica que el panel concilie las teselas publicadas en Silver.

## 1. Configuración y manifiesto de entrada

In [1]:
import pandas as pd
from IPython.display import display


def show_values(values):
    show_rows(values.items(), ["Indicador", "Valor"])


def show_rows(rows, columns=None):
    with pd.option_context("display.max_colwidth", None):
        display(pd.DataFrame(rows, columns=columns))

REGION = "us-east-1"
OPERATIONS_BUCKET = "american-tower-dev-operations-1dd72256126674a5893dfca3df"
GOLD_BUCKET = "american-tower-dev-gold-00f4f2b4cf37bac32a9ec520ee"
SILVER_RUN_URI = "s3://american-tower-dev-operations-1dd72256126674a5893dfca3df/run_id=20261125T072542Z-e92a810559/silver/_SUCCESS.json"  # Pegar la manifest_uri de entrada.
EXECUTORS = 10
EXECUTOR_CORES = 4
EXECUTOR_MEMORY = "8g"
EXECUTOR_OVERHEAD = "3g"
SHUFFLE_PARTITIONS = EXECUTORS * 12

show_values({"Región": REGION, "Manifiesto de entrada": SILVER_RUN_URI or "Pendiente de configurar", "Bucket Gold": GOLD_BUCKET, "Bucket Operations": OPERATIONS_BUCKET, "Executors solicitados": EXECUTORS})

,Indicador,Valor
0,Región,us-east-1
1,Manifiesto de entrada,s3://american-tower-dev-operations-1dd72256126674a5893dfca3df/run_id=20261125T072542Z-e92a810559/silver/_SUCCESS.json
2,Bucket Gold,american-tower-dev-gold-00f4f2b4cf37bac32a9ec520ee
3,Bucket Operations,american-tower-dev-operations-1dd72256126674a5893dfca3df
4,Executors solicitados,10


## 2. Linaje y tablas Silver

In [2]:
import json
import uuid
from datetime import datetime, timezone

import boto3
from botocore.config import Config
from botocore.exceptions import ClientError

s3 = boto3.client("s3", region_name=REGION, config=Config(retries={"mode": "standard", "max_attempts": 5}))
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:10]


def s3_path(uri):
    if not isinstance(uri, str) or not uri.startswith("s3://"):
        raise ValueError(f"Se requiere una URI S3 completa: {uri!r}")
    bucket, slash, key = uri.removeprefix("s3://").partition("/")
    if not bucket or not slash or not key:
        raise ValueError(f"Se requiere una URI S3 completa: {uri}")
    return bucket, key


def missing_object(error):
    return error.response["Error"]["Code"] in {"NoSuchKey", "NotFound", "404"}


def read_json(uri):
    bucket, key = s3_path(uri)
    try:
        response = s3.get_object(Bucket=bucket, Key=key)
    except ClientError as error:
        if missing_object(error):
            raise FileNotFoundError(f"No existe el manifiesto {uri}") from error
        raise
    with response["Body"] as stream:
        return json.load(stream)


def save_json(uri, value):
    bucket, key = s3_path(uri)
    s3.put_object(
        Bucket=bucket, Key=key, IfNoneMatch="*", ContentType="application/json",
        Body=json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False).encode("utf-8"),
    )
    return uri



def read_manifest(uri, schema, parameter):
    if not uri:
        raise ValueError(f"Completar {parameter} con la manifest_uri que devolvió la etapa anterior.")
    document = read_json(uri)
    if document.get("schema") != schema or document.get("complete") is not True or not document.get("run_id"):
        raise ValueError(f"Manifiesto incompleto o de un tipo diferente a {schema}: {uri}")
    return document
silver = read_manifest(SILVER_RUN_URI, "silver-autonomo-v1", "SILVER_RUN_URI")
tables = silver["tables"]
show_values({"Corrida Silver": silver["run_id"], "Corrida Bronze": silver["bronze_run_id"],
             "Distritos Silver": silver["distritos"], "Trimestres Silver": silver["trimestres"],
             "Teselas asignadas": silver["quality"]["espacial"]["asignadas"]})
show_rows([{"tabla": name, "uri": uri} for name, uri in tables.items()])

/opt/american-tower/venv/lib64/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,Indicador,Valor
0,Corrida Silver,20261125T072542Z-e92a810559
1,Corrida Bronze,20261125T071603Z-190d23d89e
2,Distritos Silver,1890
3,Trimestres Silver,29
4,Teselas asignadas,417420


,tabla,uri
0,distritos,s3://american-tower-dev-silver-2ad71fded155d2bd52df8f836e/run_id=20261125T072542Z-e92a810559/distritos
1,trimestres,s3://american-tower-dev-silver-2ad71fded155d2bd52df8f836e/run_id=20261125T072542Z-e92a810559/trimestres
2,osiptel_cobertura,s3://american-tower-dev-silver-2ad71fded155d2bd52df8f836e/run_id=20261125T072542Z-e92a810559/osiptel_cobertura
3,worldpop_2020,s3://american-tower-dev-silver-2ad71fded155d2bd52df8f836e/run_id=20261125T072542Z-e92a810559/worldpop_2020
4,ookla_teselas,s3://american-tower-dev-silver-2ad71fded155d2bd52df8f836e/run_id=20261125T072542Z-e92a810559/ookla_teselas


## 3. Inicialización de Spark en YARN

In [3]:
import os
import sys
from contextlib import contextmanager
from pathlib import Path

SPARK_HOME = Path("/usr/lib/spark")
py4j = sorted((SPARK_HOME / "python/lib").glob("py4j-*.zip"))
if not py4j:
    raise RuntimeError("Abrir esta notebook en el nodo primario de Amazon EMR con Spark instalado.")
os.environ["SPARK_HOME"] = str(SPARK_HOME)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
sys.path[:0] = [str(SPARK_HOME / "python"), str(py4j[0])]

from pyspark import SparkContext
from pyspark.sql import SparkSession, functions as F


@contextmanager
def spark_session(stage):
    if SparkContext._active_spark_context is not None:
        raise RuntimeError("Reiniciar el kernel para que YARN aplique los parámetros de esta corrida.")
    # EMRFS necesita un prefijo existente para el registro de eventos.
    log_dir = f"run_id={RUN_ID}/eventos_spark/"
    s3.put_object(Bucket=OPERATIONS_BUCKET, Key=log_dir, Body=b"")
    spark = (
        SparkSession.builder.master("yarn").appName(f"american-tower-{stage}")
        .config("spark.submit.deployMode", "client")
        .config("spark.pyspark.python", sys.executable)
        .config("spark.executorEnv.PYSPARK_PYTHON", sys.executable)
        .config("spark.dynamicAllocation.enabled", "false")
        .config("spark.executor.instances", str(EXECUTORS))
        .config("spark.executor.cores", str(EXECUTOR_CORES))
        .config("spark.executor.memory", EXECUTOR_MEMORY)
        .config("spark.executor.memoryOverhead", EXECUTOR_OVERHEAD)
        .config("spark.sql.shuffle.partitions", str(SHUFFLE_PARTITIONS))
        .config("spark.sql.session.timeZone", "UTC")
        .config("spark.eventLog.enabled", "true")
        .config("spark.eventLog.dir", f"s3://{OPERATIONS_BUCKET}/{log_dir}")
        .getOrCreate()
    )
    try:
        spark.sparkContext.setLogLevel("WARN")
        print(f"Aplicación Spark {spark.sparkContext.applicationId}", flush=True)
        yield spark
    finally:
        spark.stop()


def write_table(frame, uri, partitions=()):
    writer = frame.write.mode("errorifexists").option("compression", "snappy")
    if partitions:
        writer = writer.partitionBy(*partitions)
    writer.parquet(uri)
    return uri

from contextlib import ExitStack
from IPython import get_ipython
from urllib.parse import quote
from urllib.request import urlopen


def executor_usage(sc):
    if not sc.uiWebUrl:
        return {"estado": "interfaz Spark deshabilitada", "activos": None, "tareas": {}, "hosts": {}}
    app = quote(sc.applicationId, safe="")
    try:
        with urlopen(f"{sc.uiWebUrl.rstrip('/')}/api/v1/applications/{app}/allexecutors", timeout=4) as response:
            rows = [row for row in json.load(response) if row["id"] != "driver"]
    except (OSError, ValueError, KeyError) as error:
        return {"estado": f"interfaz no disponible ({type(error).__name__})",
                "activos": None, "tareas": {}, "hosts": {}}
    return {"estado": "lectura disponible", "activos": sum(row.get("isActive", False) for row in rows),
            "tareas": {row["id"]: row.get("totalTasks", 0) for row in rows},
            "hosts": {row["id"]: row.get("hostPort", "").rsplit(":", 1)[0] for row in rows}}


def show_spark(spark, events_uri):
    sc = spark.sparkContext
    show_values({"Aplicación": sc.applicationId, "Spark": spark.version, "Master": sc.master,
                 "Paralelismo base": sc.defaultParallelism, "Executors solicitados": EXECUTORS,
                 "vCores por executor": EXECUTOR_CORES,
                 "Heap y overhead por executor": f"{EXECUTOR_MEMORY} + {EXECUTOR_OVERHEAD}",
                 "Particiones shuffle": spark.conf.get("spark.sql.shuffle.partitions"),
                 "Eventos Spark": events_uri})


def open_stage(stage):
    events = get_ipython().events
    resources = ExitStack()

    def stop_on_failure(result):
        if result.error_before_exec or result.error_in_exec:
            resources.close()

    events.register("post_run_cell", stop_on_failure)
    resources.callback(events.unregister, "post_run_cell", stop_on_failure)
    try:
        spark = resources.enter_context(spark_session(stage))
    except BaseException:
        resources.close()
        raise
    events_uri = f"s3://{OPERATIONS_BUCKET}/run_id={RUN_ID}/eventos_spark/"
    return spark, resources, events_uri

spark, resources, events_uri = open_stage("gold")
show_spark(spark, events_uri)

Nov 25, 2026 7:33:57 AM org.apache.spark.launcher.Log4jHotPatchOption staticJavaAgentOption

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/25 07:34:01 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.


Aplicación Spark application_1795587001536_0005


,Indicador,Valor
0,Aplicación,application_1795587001536_0005
1,Spark,3.5.0-amzn-1
2,Master,yarn
3,Paralelismo base,2
4,Executors solicitados,10
5,vCores por executor,4
6,Heap y overhead por executor,8g + 3g
7,Particiones shuffle,120
8,Eventos Spark,s3://american-tower-dev-operations-1dd72256126674a5893dfca3df/run_id=20261125T073334Z-7ea4f607ff/eventos_spark/


## 4. Panel distrito-trimestre y revisión de faltantes

La velocidad se pondera por las pruebas de cada tesela. La falta de mediciones conserva el valor nulo.

In [4]:
def district_panel(spark, tiles, districts, periods, silver, gold_root):
    stats = tiles.groupBy("ubigeo", "period_id").agg(
        (F.sum(F.col("avg_d_kbps").cast("double") * F.col("tests")) /
         F.sum("tests") / 1000).alias("vel_descarga_mbps"),
        F.sum("tests").alias("tests"),
        F.count("*").alias("teselas_observadas"),
    )
    panel = (districts.crossJoin(periods).join(stats, ["ubigeo", "period_id"], "left")
             .withColumn("year", F.floor(F.col("period_id") / 4).cast("int"))
             .withColumn("quarter", (F.col("period_id") % 4 + 1).cast("int"))
             .withColumn("observado", F.col("vel_descarga_mbps").isNotNull()))
    uri = write_table(panel, f"{gold_root}/panel_distrital", ["year"])
    counts = spark.read.parquet(uri).agg(
        F.count("*").alias("filas"),
        F.count("vel_descarga_mbps").alias("observadas"),
        F.count_distinct("ubigeo").alias("distritos"),
        F.count_distinct("period_id").alias("trimestres"),
        F.count_distinct(F.struct("ubigeo", "period_id")).alias("claves"),
        F.sum("teselas_observadas").alias("teselas"),
    ).first().asDict()
    if (counts["distritos"] != silver["distritos"] or counts["trimestres"] != silver["trimestres"]
            or counts["filas"] != counts["distritos"] * counts["trimestres"]
            or counts["filas"] != counts["claves"] or not counts["observadas"]
            or counts["teselas"] != silver["quality"]["espacial"]["asignadas"]):
        raise ValueError("El panel Gold no concilia el grano distrito-trimestre o las teselas publicadas en Silver.")
    return uri, counts

gold_root = f"s3://{GOLD_BUCKET}/run_id={RUN_ID}"
ops_root = f"s3://{OPERATIONS_BUCKET}/run_id={RUN_ID}/gold"
districts = spark.read.parquet(tables["distritos"])
periods = spark.read.parquet(tables["trimestres"])
tiles = spark.read.parquet(tables["ookla_teselas"])
panel_uri, panel = district_panel(spark, tiles, districts, periods, silver, gold_root)
show_values({**panel, "sin_medición": panel["filas"] - panel["observadas"], "Panel": panel_uri})
panel_df = spark.read.parquet(panel_uri)
(panel_df.groupBy("year", "quarter").agg(F.count("*").alias("universo"),
    F.count("vel_descarga_mbps").alias("observadas")).orderBy("year", "quarter").show(panel["trimestres"], truncate=False))

,Indicador,Valor
0,filas,54810
1,observadas,41434
2,distritos,1890
3,trimestres,29
4,claves,54810
5,teselas,417420
6,sin_medición,13376
7,Panel,s3://american-tower-dev-gold-00f4f2b4cf37bac32a9ec520ee/run_id=20261125T073334Z-7ea4f607ff/panel_distrital


+----+-------+--------+----------+
|year|quarter|universo|observadas|
+----+-------+--------+----------+
|2019|1      |1890    |1004      |
|2019|2      |1890    |1034      |
|2019|3      |1890    |1138      |
|2019|4      |1890    |1182      |
|2020|1      |1890    |1231      |
|2020|2      |1890    |1111      |
|2020|3      |1890    |1394      |
|2020|4      |1890    |1453      |
|2021|1      |1890    |1381      |
|2021|2      |1890    |1465      |
|2021|3      |1890    |1496      |
|2021|4      |1890    |1472      |
|2022|1      |1890    |1498      |
|2022|2      |1890    |1474      |
|2022|3      |1890    |1503      |
|2022|4      |1890    |1497      |
|2023|1      |1890    |1475      |
|2023|2      |1890    |1486      |
|2023|3      |1890    |1455      |
|2023|4      |1890    |1486      |
|2024|1      |1890    |1499      |
|2024|2      |1890    |1527      |
|2024|3      |1890    |1613      |
|2024|4      |1890    |1645      |
|2025|1      |1890    |1572      |
|2025|2      |1890  

## 5. Contexto distrital de población y cobertura declarada

In [5]:
def district_context(spark, districts, population, coverage, silver, gold_root):
    # Una fila por centro poblado. Si al menos una operadora declara 4G, el centro cuenta como positivo.
    towns = (coverage.filter("centro_poblado IS NOT NULL AND ubigeo IS NOT NULL")
             .groupBy("ubigeo", "centro_poblado").agg(F.max("declara_4g").alias("alguna_4g")))
    district_cov = towns.groupBy("ubigeo").agg(
        F.count("*").alias("centros_en_reporte"),
        F.count("alguna_4g").alias("centros_con_dato_4g"),
        F.sum("alguna_4g").alias("centros_con_4g_declarada"),
    ).withColumn("periodo_referencia", F.lit(silver["osiptel_periodo"])).withColumn(
        "fecha_corte_fuente", F.to_date(F.lit(silver["osiptel_fecha_corte"])))
    context = districts.join(population, "ubigeo", "left").join(district_cov, "ubigeo", "left")
    uri = write_table(context, f"{gold_root}/contexto_territorial")
    counts = spark.read.parquet(uri).agg(
        F.count("*").alias("filas"), F.count_distinct("ubigeo").alias("claves"),
        F.count("poblacion_2020").alias("con_poblacion"), F.count("centros_en_reporte").alias("con_osiptel"),
    ).first().asDict()
    if (counts["filas"] != silver["distritos"] or counts["filas"] != counts["claves"]
            or not counts["con_poblacion"] or not counts["con_osiptel"]):
        raise ValueError("El contexto Gold no concilia la dimensión distrital, WorldPop o la cobertura OSIPTEL.")
    return uri, counts

population = spark.read.parquet(tables["worldpop_2020"])
coverage = spark.read.parquet(tables["osiptel_cobertura"])
context_uri, context = district_context(spark, districts, population, coverage, silver, gold_root)
show_values({**context, "Referencia de población": 2020, "Corte OSIPTEL": silver["osiptel_fecha_corte"], "Contexto": context_uri})
spark.read.parquet(context_uri).select("ubigeo", "distrito", "poblacion_2020", "centros_con_dato_4g",
                                  "centros_con_4g_declarada").show(8, truncate=False)

,Indicador,Valor
0,filas,1890
1,claves,1890
2,con_poblacion,1890
3,con_osiptel,1663
4,Referencia de población,2020
5,Corte OSIPTEL,2023-06-13
6,Contexto,s3://american-tower-dev-gold-00f4f2b4cf37bac32a9ec520ee/run_id=20261125T073334Z-7ea4f607ff/contexto_territorial


+------+-----------+------------------+-------------------+------------------------+
|ubigeo|distrito   |poblacion_2020    |centros_con_dato_4g|centros_con_4g_declarada|
+------+-----------+------------------+-------------------+------------------------+
|100313|PACHAS     |14212.787638125941|52                 |31                      |
|100316|QUIVILLA   |4080.154700808227 |24                 |11                      |
|100317|RIPAN      |7610.267654880881 |12                 |9                       |
|100321|SHUNQUI    |2391.0516580343246|25                 |14                      |
|100322|SILLAPATA  |2330.883949354291 |43                 |32                      |
|100323|YANAS      |2967.5522934533656|47                 |34                      |
|100401|HUACAYBAMBA|7148.847587696277 |14                 |1                       |
|100402|CANCHABAMBA|3270.8470248430967|5                  |0                       |
+------+-----------+------------------+-------------------+------

## 6. Publicación Gold

In [6]:
usage = executor_usage(spark.sparkContext)
manifest = {"panel_uri": panel_uri, "panel": panel, "context_uri": context_uri, "context": context,
            "schema": "gold-autonomo-v2", "complete": True, "run_id": RUN_ID,
            "silver_run_id": silver["run_id"], "silver_manifest_uri": SILVER_RUN_URI,
            "bronze_run_id": silver["bronze_run_id"], "bronze_manifest_uri": silver["bronze_manifest_uri"],
            "application_id": spark.sparkContext.applicationId, "spark_event_log_uri": events_uri,
            "completed_at_utc": datetime.now(timezone.utc).isoformat()}
resultado = {**manifest, "manifest_uri": save_json(f"{ops_root}/_SUCCESS.json", manifest)}
show_values({"Panel distrito-trimestre": panel_uri, "Contexto distrital": context_uri,
             "Executors activos al final": usage["activos"], "Estado del monitoreo": usage["estado"],
             "Manifiesto Gold": resultado["manifest_uri"]})
print(f"GOLD_RUN_URI = {resultado['manifest_uri']!r}")
resources.close()

,Indicador,Valor
0,Panel distrito-trimestre,s3://american-tower-dev-gold-00f4f2b4cf37bac32a9ec520ee/run_id=20261125T073334Z-7ea4f607ff/panel_distrital
1,Contexto distrital,s3://american-tower-dev-gold-00f4f2b4cf37bac32a9ec520ee/run_id=20261125T073334Z-7ea4f607ff/contexto_territorial
2,Executors activos al final,8
3,Estado del monitoreo,lectura disponible
4,Manifiesto Gold,s3://american-tower-dev-operations-1dd72256126674a5893dfca3df/run_id=20261125T073334Z-7ea4f607ff/gold/_SUCCESS.json


GOLD_RUN_URI = 's3://american-tower-dev-operations-1dd72256126674a5893dfca3df/run_id=20261125T073334Z-7ea4f607ff/gold/_SUCCESS.json'
